# 01 — Local and Hosted Model Gateway

**FinAI Academy — Arnaud Demes**

Build the first working layer of the Financial Analyst Copilot: one model call that can run locally with Ollama or through OpenAI without changing the lesson code.

## Learning objectives

By the end of this notebook, you can:

- explain the message contract of a chat model;
- distinguish a model from the application built around it;
- configure Ollama or OpenAI through the same Python boundary;
- measure model latency and retain run metadata;
- expose why a successful API call can still produce a poor financial answer; and
- identify which model choices belong in configuration rather than notebook code.

## Where this fits

This is the first executable layer of the capstone. Today the model receives a question and returns text. The next notebook will turn that text into a validated financial object. Later notebooks will supply document evidence, tools, memory, an agent loop, MCP capabilities, and evaluation.

```text
Question → Settings → Model gateway → Response
```

> **Important:** a language model is a component. The Financial Analyst Copilot is the complete system that controls its context, tools, outputs, evidence and evaluation.

## One configuration contract

Every LLM-dependent notebook reads the same environment variables:

| Variable | Local example | Hosted example |
|---|---|---|
| `FINAI_MODEL_PROVIDER` | `ollama` | `openai` |
| `FINAI_CHAT_MODEL` | `qwen3:8b` | `gpt-5-mini` |
| `FINAI_EMBEDDING_PROVIDER` | `ollama` | `openai` |
| `FINAI_EMBEDDING_MODEL` | `qwen3-embedding:0.6b` | `text-embedding-3-small` |

The provider adapters change. The lesson logic does not. API keys stay in the environment and never appear in notebook cells.

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from time import perf_counter
from typing import Any

from finai_academy.providers import ModelRun, create_chat_model, provider_summary
from finai_academy.settings import Settings

### Live mode and test mode

When you open the notebook normally, `FINAI_LIVE_MODE` defaults to `1` and the configured provider is called. The automated course suite sets it to `0` and uses a deterministic recorded response. That makes regression tests fast and free while preserving separate live Ollama and OpenAI acceptance runs.

The recorded response is a test double, not a claim that a provider was contacted.

In [ ]:
@dataclass(frozen=True)
class RecordedMessage:
    content: str
    response_metadata: dict[str, Any]


class RecordedChatModel:
    """Deterministic response used only by the offline execution suite."""

    def invoke(self, messages: list[tuple[str, str]]) -> RecordedMessage:
        question = messages[-1][1]
        if "source excerpt" in question.casefold():
            content = (
                "The excerpt describes stronger data-centre demand, but it does not "
                "identify the company, reporting period or quantified financial impact. "
                "Those points remain open questions."
            )
        else:
            content = (
                "AI demand can support growth, but the question does not specify a company, "
                "period, source or required evidence."
            )
        return RecordedMessage(content=content, response_metadata={"mode": "offline fixture"})

In [ ]:
settings = Settings.from_environment()
live_mode = os.getenv("FINAI_LIVE_MODE", "1") == "1"
model = create_chat_model(settings) if live_mode else RecordedChatModel()

print("Execution mode:", "live" if live_mode else "offline fixture")
print("Provider configuration:", provider_summary(settings))

## The chat message contract

A chat call is an ordered list of messages, not one magical prompt string. The two roles used here have different responsibilities:

- **system** — stable application instructions and boundaries;
- **human** — the current request and its data.

Later we will add tool messages and conversation state. Keeping roles explicit makes prompts easier to inspect, test and version.

In [ ]:
first_messages = [
    ("system", "You are a careful financial analyst assistant."),
    ("human", "What is happening with AI demand?"),
]
first_messages

## Measure the call

Latency, provider and model are application data. Capturing them at the gateway gives later notebooks a consistent place to add token usage, cost and trace identifiers.

In [ ]:
def invoke_with_metrics(
    chat_model: Any,
    messages: list[tuple[str, str]],
    configured_settings: Settings,
) -> tuple[ModelRun, Any]:
    started = perf_counter()
    response = chat_model.invoke(messages)
    latency_ms = (perf_counter() - started) * 1_000
    run = ModelRun(
        provider=configured_settings.provider if live_mode else "offline",
        model=configured_settings.chat_model if live_mode else "recorded-response-v1",
        text=str(response.content),
        latency_ms=latency_ms,
    )
    return run, response

In [ ]:
first_run, first_response = invoke_with_metrics(model, first_messages, settings)
print(first_run.text)
print(f"\nprovider={first_run.provider} model={first_run.model} latency={first_run.latency_ms:.0f} ms")

### Inspect more than the text

Provider SDKs often return finish reasons, token usage, model identifiers and safety metadata beside the answer. The exact fields vary, which is why the application normalizes only the metadata it truly needs.

In [ ]:
getattr(first_response, "response_metadata", {})

## What the main parameters control

- **Model** changes capability, latency, price and local hardware requirements.
- **Temperature** changes sampling behaviour; lower is useful for repeatable analytical tasks but does not guarantee factuality.
- **Maximum output tokens** limits generation length, not the amount of evidence the model can read.
- **Context window** is shared by instructions, conversation, documents, tool results and the generated answer.

Different model families expose different sampling controls. The gateway should not force a parameter that the selected provider or model does not support.

## Failure lab

The first call can return fluent text, yet the request is impossible to answer professionally. It omitted:

1. the company;
2. the reporting period;
3. the source material;
4. the evidence standard; and
5. the required output.

A model cannot infer an application contract reliably from `What is happening with AI demand?`. The problem is not solved by selecting a larger model.

## Improve the request before adding infrastructure

We still do not have RAG. We can nevertheless make the task explicit and provide a small source excerpt directly. Notice how instructions and untrusted source data remain separated.

In [ ]:
source_excerpt = (
    "Management stated that demand for data-centre systems strengthened during the period, "
    "while supply availability remained a constraint."
)

grounded_messages = [
    (
        "system",
        "You are a careful financial analyst assistant. Use only the supplied source. "
        "Separate what the source states from your interpretation and identify missing information.",
    ),
    (
        "human",
        f"Analyse the source excerpt. Return the supported development and the open questions.\n\n"
        f"<source_excerpt>\n{source_excerpt}\n</source_excerpt>",
    ),
]

In [ ]:
grounded_run, _ = invoke_with_metrics(model, grounded_messages, settings)
print(grounded_run.text)
print(f"\nlatency={grounded_run.latency_ms:.0f} ms")

### Debrief

The improved call is still only text. We have not validated its shape, checked every claim, or retrieved from a long document. That is intentional: each limitation creates the next lesson.

The engineering improvement in this notebook is the **provider boundary**. The prompt improvement previews Notebook 02.

## Switch providers without changing code

Start Jupyter with one of these configurations and run the notebook from the top:

```bash
FINAI_MODEL_PROVIDER=ollama FINAI_CHAT_MODEL=qwen3:8b uv run --extra ai jupyter lab
```

```bash
FINAI_MODEL_PROVIDER=openai FINAI_CHAT_MODEL=gpt-5-mini OPENAI_API_KEY=... uv run --extra ai jupyter lab
```

For the capstone demonstration, `FINAI_CHAT_MODEL=gpt-5.1` is a configuration change, not a code change.

## Verification

These assertions check the notebook’s own output contract. They do not claim that a response is financially correct; later evaluation notebooks add that layer.

In [ ]:
assert first_run.text.strip(), "The model response must not be empty."
assert grounded_run.text.strip(), "The grounded response must not be empty."
assert first_run.latency_ms >= 0
assert grounded_run.latency_ms >= 0
assert settings.provider in {"ollama", "openai"}
print("PASS — provider-neutral model gateway verified")

## Challenge

Run the notebook once with Ollama and once with OpenAI. Record:

- model identifier;
- latency for each call;
- one material difference in the grounded answer;
- one privacy or cost trade-off; and
- whether that difference justifies provider-specific application code.

Your conclusion should normally preserve the shared boundary even when one provider performs better.

## Capstone integration

The capstone now owns four reusable elements:

1. `Settings.from_environment()` for provider-specific defaults;
2. `create_chat_model(settings)` for lazy provider construction;
3. `provider_summary(settings)` for safe diagnostics; and
4. `ModelRun` for normalized run metadata.

Notebook 02 will replace free-form text with a Pydantic-validated `AnalystBrief`.

## Recap

- A model call is an ordered message exchange.
- A model is only one component of an AI application.
- Provider choice belongs behind a small application boundary.
- Ollama and OpenAI can run the same notebook code.
- Fluent text is not yet a reliable financial product.
- The next engineering need is a validated output contract.